In [1]:
# import libraries

import requests
import pandas as pd
import os

/Users/helloklow/pet-adoption-predictor/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# fetch data from the api

url = "https://data.austintexas.gov/resource/9t4d-g238.json"

params = {
    "$limit": 10000, # 10000 record limit
    "$offset": 0 # start from record 0
}

response = requests.get(url, params=params)
print(response.status_code)

200


In [3]:
# parse response in df

data = response.json()
df = pd.DataFrame(data)
print(df.shape)
print(df.head())

(10000, 12)
  animal_id date_of_birth                   datetime monthyear outcome_type  \
0   A668305    2012-12-01  2013-12-02T00:00:00-05:00   12-2013     Transfer   
1   A673335    2012-02-22  2014-02-22T00:00:00-05:00   02-2014   Euthanasia   
2   A675999    2013-04-03  2014-04-07T00:00:00-05:00   04-2014     Transfer   
3   A679066    2014-04-16  2014-05-16T00:00:00-05:00   05-2014          NaN   
4   A680855    2014-05-25  2014-06-10T00:00:00-05:00   06-2014     Transfer   

  outcome_subtype animal_type sex_upon_outcome age_upon_outcome       breed  \
0         Partner       Other          Unknown           1 year  Turtle Mix   
1       Suffering       Other          Unknown          2 years     Raccoon   
2         Partner       Other          Unknown           1 year  Turtle Mix   
3             NaN       Other          Unknown          4 weeks   Rabbit Sh   
4         Partner        Bird          Unknown          2 weeks        Duck   

          color name  
0  Brown/Yellow

In [4]:
# check columns

print(df.columns.tolist())

['animal_id', 'date_of_birth', 'datetime', 'monthyear', 'outcome_type', 'outcome_subtype', 'animal_type', 'sex_upon_outcome', 'age_upon_outcome', 'breed', 'color', 'name']


In [5]:
# check outcome_type
# transfer is most common, followed by adoption
# euthanasia is a significant outcome
# remaining outcome types are sparse; consider grouping or dropping

print(df['outcome_type'].value_counts())

outcome_type
Transfer           4216
Adoption           3266
Return to Owner    1533
Euthanasia          865
Died                 64
Disposal             34
Missing              16
Relocate              2
Rto-Adopt             1
Name: count, dtype: int64


In [6]:
# target variable: binary classification problem
# 1 = adopted, 0 = not adopted

In [7]:
# pull all the data

all_data = []
limit = 10000
offset = 0

# loop to fetch all records
while True:
    params = {"$limit": limit, "$offset": offset}
    response = requests.get(url, params=params)
    batch = response.json()

    # if batch is empty, fetching is complete, stop looping
    if len(batch) == 0:
        break
        
    all_data.extend(batch) # add batch to all_data
    offset += limit # move offset forward
    print(f"Fetched {offset} records so far...") # print progress

df = pd.DataFrame(all_data) # convert all_data to pandas df
print(f"Total records: {df.shape}")
    

Fetched 10000 records so far...
Fetched 20000 records so far...
Fetched 30000 records so far...
Fetched 40000 records so far...
Fetched 50000 records so far...
Fetched 60000 records so far...
Fetched 70000 records so far...
Fetched 80000 records so far...
Fetched 90000 records so far...
Fetched 100000 records so far...
Fetched 110000 records so far...
Fetched 120000 records so far...
Fetched 130000 records so far...
Fetched 140000 records so far...
Fetched 150000 records so far...
Fetched 160000 records so far...
Fetched 170000 records so far...
Fetched 180000 records so far...
Total records: (173775, 12)


In [8]:
# save data locally (reduce hitting api)

# create the data directory if it doesn't exist
os.makedirs("../data", exist_ok=True)

# save the raw data as a csv (never modify)
df.to_csv("../data/raw_adoptions.csv", index=False)

print("Data saved successfully!")

Data saved successfully!
